# **Step 2a – Creación del dataset objetivo: consolidación del Bloque 2 (Nutrición 2026)**
**Proyecto TA – Análisis de Datos (1INF03, 2026-2) – Equipo E10**

> *KDD, paso 2:* «Creating a target data set: selecting a data set, or focusing on a subset of variables or data samples, on which discovery is to be performed.»

Las mediciones llegan en **6 Excel mensuales** con una hoja por hogar. Las columnas **no están en la misma posición** en todas las hojas (algunas traen P/E, otras presión arterial o perímetro cefálico, y agosto trae dos columnas de edad). Este notebook une todo en una sola tabla de forma **trazable y verificable**.

Entrada: `datos/0_crudos/nutricion_mensual/*.xlsx`.
Salidas: `datos/1_consolidados/step2a_nutricion_2026.csv` y `step2a_control_origen.csv`.

## Librerías y rutas

In [1]:
import re, json, unicodedata
from pathlib import Path
import numpy as np, pandas as pd
import openpyxl
RAW = Path("../datos/0_crudos/nutricion_mensual")
OUT = Path("../datos/1_consolidados"); RES = Path("../datos/4_resultados")
ANTIGUO = Path("../../Output/datos/raw/Nutricion_Consolidado_2026.xlsx")   # consolidado previo, solo para verificar

def norm(s):
    """Texto comparable: sin tildes, mayúsculas y espacios/saltos de línea colapsados."""
    s = unicodedata.normalize("NFKD", str(s))
    return re.sub(r"\s+", " ", "".join(c for c in s if not unicodedata.combining(c))).strip().upper()

## 1. Mapeo explícito de columnas por hoja
Se escribe a mano, para cada archivo y hoja, en qué columna (numeración de Excel: A = 1) está cada variable. Se prefirió un mapeo explícito a uno automático para que cualquier cambio de posición sea visible y revisable.

- La fila de cabecera es la 3 en todas las hojas; los datos empiezan debajo.
- Los diagnósticos se agrupan en dos secciones: **OMS** (percentiles / puntaje z) e **INS** (tabla de diagnóstico rápido).
- `EDAD` y `EDAD C/ FORMULA` **no** se mapean (ver §4).

In [2]:
D_INF = {"id": 2, "nac": 3, "peso": 5, "talla": 6, "imc": 7,
         "P/E_OMS": 8, "T/E_OMS": 9, "P/T_OMS": 10, "IMC_OMS": 11, "P/E_INS": 12, "T/E_INS": 13, "P/T_INS": 14, "IMC_INS": 15}
D_MAY_ENE = {"id": 2, "nac": 3, "peso": 5, "talla": 6, "imc": 7, "T/E_OMS": 8, "IMC_OMS": 9, "T/E_INS": 10, "IMC_INS": 11}
D_MAY_PE  = {"id": 2, "nac": 3, "peso": 5, "talla": 6, "imc": 7, "P/E_OMS": 8, "T/E_OMS": 9, "IMC_OMS": 10, "T/E_INS": 11, "IMC_INS": 12}
MAPEO = {
    ("2026_01", "INFANTES"): D_INF,
    ("2026_01", "MIGUEL ECHENIQUE"): D_MAY_ENE, ("2026_01", "TOMAS VALLE"): D_MAY_ENE,
    ("2026_01", "MATOGROSO MUJERES"): D_MAY_ENE, ("2026_01", "MATOGROSO HOMBRES"): D_MAY_ENE,
    ("2026_02", "Eva.Nut febrero"): D_INF,
    ("2026_03", "Eva.Nut marzo"): {**{k: v for k, v in D_INF.items() if k != "imc"}, "pc": 7},
    ("2026_04", "INFANTES (ABR)"): D_INF,
    ("2026_04", "MIGUEL ECHENIQUE"): {"id": 2, "nac": 3, "peso": 5, "talla": 6, "pa": 7, "imc": 8,
                                      "T/E_OMS": 9, "IMC_OMS": 10, "T/E_INS": 11, "IMC_INS": 12},
    ("2026_04", "TOMAS VALLE"): {"id": 2, "nac": 3, "peso": 5, "talla": 6, "pa": 7, "imc": 8,
                                 "P/E_OMS": 9, "T/E_OMS": 10, "IMC_OMS": 11, "T/E_INS": 12, "IMC_INS": 13},
    ("2026_04", "MATOGROSO MUJERES"): D_MAY_PE, ("2026_04", "MATOGROSO HOMBRES"): D_MAY_PE,
    ("2026_06", "INFANTES "): D_INF,
    ("2026_06", "MIGUEL ECHENIQUE"): D_MAY_PE, ("2026_06", "TOMAS VALLE"): D_MAY_PE,
    ("2026_06", "MATOGROSO MUJERES"): D_MAY_PE, ("2026_06", "MATOGROSO HOMBRES"): D_MAY_PE,
    ("2026_08", "MATOGROSO MUJERES"): {"id": 2, "nac": 3, "peso": 6, "talla": 7, "imc": 8,
                                       "T/E_OMS": 9, "IMC_OMS": 10, "T/E_INS": 11, "IMC_INS": 12},
    ("2026_08", "MATOGROSO HOMBRES"): {"id": 2, "nac": 3, "peso": 6, "talla": 7, "imc": 8,
                                       "T/E_OMS": 9, "IMC_OMS": 10, "T/E_INS": 11, "IMC_INS": 12},
}
CASA = {"INFANTES": "INFANTES", "INFANTES (ABR)": "INFANTES", "INFANTES ": "INFANTES",
        "Eva.Nut febrero": "INFANTES", "Eva.Nut marzo": "INFANTES"}
print("Hojas mapeadas:", len(MAPEO))

Hojas mapeadas: 19


## 2. Verificación del mapeo contra las cabeceras
Para cada hoja se comprueba que el texto de la cabecera en cada columna mapeada sea el esperado (p. ej. la columna de `peso` debe decir `PESO (kg)`) y que cada diagnóstico esté en la sección correcta (OMS antes que INS). Si una columna estuviera corrida, el notebook se detiene aquí.

In [3]:
ESPERADO = {"id": {"APELLIDOS Y NOMBRES"}, "nac": {"FECHA DE NACIMIENTO"}, "peso": {"PESO (KG)"}, "talla": {"TALLA (CM)"},
            "imc": {"IMC"}, "pc": {"P.CEFALICO"}, "pa": {"PA", "P.A"}}
for ind in ["P/E", "T/E", "P/T"]:
    for ref in ["OMS", "INS"]: ESPERADO[f"{ind}_{ref}"] = {ind}
for ref in ["OMS", "INS"]: ESPERADO[f"IMC_{ref}"] = {"IMC", "IMC/E"}

archivos = {f.name[14:18] + "_" + f.name[18:20]: f for f in sorted(RAW.glob("*.xlsx"))}
libros = {p: openpyxl.load_workbook(f, data_only=True) for p, f in archivos.items()}
assert set(h for _, h in MAPEO) <= {ws.title for wb in libros.values() for ws in wb.worksheets}
assert len(MAPEO) == sum(len(wb.worksheets) for wb in libros.values()), "hay hojas sin mapear"

revision = []
for (per, hoja), mp in MAPEO.items():
    ws = libros[per][hoja]
    cab = {j: norm(ws.cell(3, j).value) for j in mp.values()}
    for var, j in mp.items():
        assert cab[j] in ESPERADO[var], f"{per}/{hoja}: columna {j} ({var}) dice '{cab[j]}'"
    # inicio de la sección INS = celda de la fila 2 que menciona 'diagnóstico rápido' o 'tabla'
    ins = [c.column for c in ws[2] if c.value and re.search(r"RAPIDO|TABLA|INS", norm(c.value))]
    ins_col = ins[0]
    oms_cols = [j for v, j in mp.items() if v.endswith("_OMS")]; ins_cols = [j for v, j in mp.items() if v.endswith("_INS")]
    assert max(oms_cols) < ins_col <= min(ins_cols), f"{per}/{hoja}: secciones OMS/INS mal asignadas"
    rotulo_oms = [norm(c.value) for c in ws[2] if c.value and c.column < ins_col and c.column > 1]
    revision.append({"periodo": per, "hoja": hoja, "rótulo OMS": rotulo_oms[0] if rotulo_oms else "(vacío)",
                     "inicio INS (col)": ins_col, "variables": len(mp)})
pd.DataFrame(revision)

,periodo,hoja,rótulo OMS,inicio INS (col),variables
0,2026_01,INFANTES,SEGUN PERCENTILES OMS,12,13
1,2026_01,MIGUEL ECHENIQUE,SEGUN PERCENTILES,10,9
2,2026_01,TOMAS VALLE,SEGUN PERCENTILES,10,9
3,2026_01,MATOGROSO MUJERES,SEGUN PERCENTILES,10,9
4,2026_01,MATOGROSO HOMBRES,SEGUN PERCENTILES,10,9
5,2026_02,Eva.Nut febrero,SEGUN PERCENTILES OMS,12,13
6,2026_03,Eva.Nut marzo,SEGUN PERCENTILES OMS,12,13
7,2026_04,INFANTES (ABR),SEGUN PERCENTILES OMS,12,13
8,2026_04,MIGUEL ECHENIQUE,(vacío),11,10
9,2026_04,TOMAS VALLE,SEGUN PERCENTIL,12,11


**Interpretación.** Las 19 hojas pasan la verificación: cada columna mapeada tiene la cabecera esperada y los diagnósticos OMS quedan antes de la sección INS. Solo en **abril – Miguel Echenique** el rótulo de la sección OMS está vacío en el original (celda combinada `I2:J2` sin texto); sus columnas se asignan a OMS por posición, igual que en todas las demás hojas, donde el par T/E–IMC previo a la tabla rápida es siempre OMS.

## 3. Lectura de las filas
Para cada hoja se toman las filas cuyo ID empieza con `PPA_FN_`. Reglas:
- Si el peso dice `PROVINCIA`, el NNA estaba fuera (en provincia) y **no tiene medidas**: se excluye y se cuenta.
- La fecha de nacimiento puede venir como fecha de Excel o como texto `DD/MM/AAAA`; se convierte a `AAAA-MM-DD`. Cualquier otro formato detiene la ejecución.
- `*` significa "sin diagnóstico" y pasa a vacío.

In [4]:
def fecha_iso(v):
    if hasattr(v, "strftime"): return v.strftime("%Y-%m-%d")
    m = re.fullmatch(r"(\d{2})/(\d{2})/(\d{4})", str(v).strip())
    if m: return pd.Timestamp(int(m[3]), int(m[2]), int(m[1])).strftime("%Y-%m-%d")
    raise ValueError(f"fecha de nacimiento no reconocida: {v!r}")

def texto(v):
    if v is None: return None
    s = re.sub(r"\s+", " ", str(v)).strip()
    return None if s in ("", "*") else s

filas, control, ddmm = [], [], []
for (per, hoja), mp in MAPEO.items():
    ws = libros[per][hoja]; n_id = n_prov = 0
    for r in ws.iter_rows(min_row=4, values_only=True):
        celda = lambda var: r[mp[var]-1] if var in mp and mp[var]-1 < len(r) else None
        idv = celda("id")
        if not (isinstance(idv, str) and idv.strip().startswith("PPA_FN_")): continue
        n_id += 1
        if norm(celda("peso")) == "PROVINCIA": n_prov += 1; continue
        if isinstance(celda("nac"), str): ddmm.append((per, idv.strip(), celda("nac")))
        d = {"periodo": per, "archivo": archivos[per].name, "hoja": hoja.strip(), "casa": CASA.get(hoja, hoja.strip()),
             "IDs_FichaNominal": idv.strip(), "nac": fecha_iso(celda("nac")),
             "Peso_kg": float(celda("peso")), "Talla_cm": float(celda("talla")),
             "IMC_valor": float(celda("imc")) if celda("imc") not in (None, "*") else np.nan,
             "Perimetro_cefalico": celda("pc") if celda("pc") not in (None, "*") else np.nan,
             "Presion_arterial": texto(celda("pa"))}
        for dx in ["P/E_OMS", "T/E_OMS", "P/T_OMS", "IMC_OMS", "P/E_INS", "T/E_INS", "P/T_INS", "IMC_INS"]:
            d[dx] = texto(celda(dx))
        filas.append(d)
    control.append({"periodo": per, "archivo": archivos[per].name, "hoja": hoja.strip(), "filas_id": n_id,
                    "filas_provincia": n_prov, "filas_consolidadas": n_id - n_prov,
                    "mapeo_columnas": ",".join(f"{k}:{v}" for k, v in mp.items())})
nut = pd.DataFrame(filas); ctrl = pd.DataFrame(control)
print("Filas con ID:", int(ctrl["filas_id"].sum()), "| PROVINCIA excluidas:", int(ctrl["filas_provincia"].sum()),
      "| consolidadas:", len(nut))
print("Fechas de nacimiento escritas como texto DD/MM/AAAA:", len(ddmm)); print(pd.DataFrame(ddmm, columns=["periodo","ID","texto"]).to_string(index=False))
ctrl[["periodo","hoja","filas_id","filas_provincia","filas_consolidadas"]]

Filas con ID: 402 | PROVINCIA excluidas: 23 | consolidadas: 379
Fechas de nacimiento escritas como texto DD/MM/AAAA: 8
periodo          ID        texto
2026_01 PPA_FN_0033   12/04/2014
2026_01 PPA_FN_0034   12/04/2014
2026_01 PPA_FN_0030   21/11/2012
2026_01 PPA_FN_0090   19/08/2015
2026_04 PPA_FN_0033   12/04/2014
2026_04 PPA_FN_0034   12/04/2014
2026_04 PPA_FN_0030   21/11/2012
2026_06 PPA_FN_0030   21/11/2012


,periodo,hoja,filas_id,filas_provincia,filas_consolidadas
0,2026_01,INFANTES,33,0,33
1,2026_01,MIGUEL ECHENIQUE,17,0,17
2,2026_01,TOMAS VALLE,22,0,22
3,2026_01,MATOGROSO MUJERES,32,4,28
4,2026_01,MATOGROSO HOMBRES,37,5,32
5,2026_02,Eva.Nut febrero,7,0,7
6,2026_03,Eva.Nut marzo,6,0,6
7,2026_04,INFANTES (ABR),30,0,30
8,2026_04,MIGUEL ECHENIQUE,16,0,16
9,2026_04,TOMAS VALLE,20,0,20


**Interpretación.** De 402 filas con ID, 23 son NNA en **provincia** sin peso ni talla (solo en los hogares Matogroso) y se excluyen; quedan **379 evaluaciones**. Ocho fechas de nacimiento estaban escritas como texto `DD/MM/AAAA` en el original (4 NNA): se convierten con su formato explícito, sin que pandas adivine el orden día/mes.

## 4. Columna de edad: por qué no se consolida
En junio (y la columna `EDAD C/ FORMULA` de agosto) la edad es una **fórmula** `DATEDIF(nacimiento; HOY())`, que se recalcula cada vez que se abre el archivo y por tanto **no** es la edad a la fecha de la evaluación. Se comprueba leyendo las fórmulas.

In [5]:
for per, hoja, celda in [("2026_06", "INFANTES ", "D5"), ("2026_08", "MATOGROSO MUJERES", "D5"), ("2026_08", "MATOGROSO MUJERES", "E5"), ("2026_01", "INFANTES", "D5")]:
    v = openpyxl.load_workbook(archivos[per])[hoja][celda].value
    print(f"{per} {hoja.strip():18s} {celda}: {v}")

2026_06 INFANTES           D5: =DATEDIF(C5,TODAY(),"y") & " años "  & DATEDIF(C5,TODAY(),"ym") & " meses "
2026_08 MATOGROSO MUJERES  D5: =DATEDIF(C5,TODAY(),"Y")&" años y "&DATEDIF(C5,TODAY(),"YM")&"meses"
2026_08 MATOGROSO MUJERES  E5: 15 años y 10meses


2026_01 INFANTES           D5: 1 año 4 meses


**Decisión.** La edad se calculará siempre desde la fecha de nacimiento y la fecha de evaluación (Step 3). Las edades escritas **a mano** se usan solo en el Step 2b para confirmar el mes de la evaluación.

## 5. IMC faltante
Marzo no trae columna de IMC. Se recalcula con `peso / (talla/100)²` y se marca con `imc_recalculado = 1`. En las filas que sí traen IMC se mide cuánto difiere del recalculado, para saber si la fórmula de la institución es la misma.

In [6]:
imc_calc = nut["Peso_kg"] / (nut["Talla_cm"]/100)**2
nut["imc_recalculado"] = nut["IMC_valor"].isna().astype(int)
dif = (nut["IMC_valor"] - imc_calc).abs()
assert (nut.loc[dif > 0.05, "periodo"] == "2026_08").all()
print("IMC faltante:", int(nut["imc_recalculado"].sum()), "| diferencia máx. donde existía:", round(dif.max(), 3), "| mediana:", round(dif.median(), 4))
print(nut.loc[dif > 0.05, ["periodo","hoja","IDs_FichaNominal","Peso_kg","Talla_cm","IMC_valor"]].assign(IMC_recalc=imc_calc.round(2)).to_string(index=False))
nut["IMC_valor"] = nut["IMC_valor"].fillna(imc_calc)

IMC faltante: 6 | diferencia máx. donde existía: 0.462 | mediana: 0.0
periodo              hoja IDs_FichaNominal  Peso_kg  Talla_cm  IMC_valor  IMC_recalc
2026_08 MATOGROSO MUJERES      PPA_FN_0027    58.30     156.4      24.20       23.83
2026_08 MATOGROSO MUJERES      PPA_FN_0032    45.00     146.6      21.40       20.94
2026_08 MATOGROSO MUJERES      PPA_FN_0043    42.90     145.7      20.10       20.21
2026_08 MATOGROSO MUJERES      PPA_FN_0045    57.70     159.1      23.08       22.79
2026_08 MATOGROSO MUJERES      PPA_FN_0048    50.20     147.4      23.20       23.11
2026_08 MATOGROSO MUJERES      PPA_FN_0047    49.45     141.6      24.60       24.66
2026_08 MATOGROSO MUJERES      PPA_FN_0062    55.50     150.9      24.30       24.37
2026_08 MATOGROSO MUJERES      PPA_FN_0061    30.25     143.5      14.60       14.69
2026_08 MATOGROSO MUJERES      PPA_FN_0044    75.40     153.5      31.80       32.00
2026_08 MATOGROSO HOMBRES      PPA_FN_0067    64.80     164.6      23.80       2

**Interpretación.** Solo faltan 6 IMC (todos de marzo, que no trae esa columna). De enero a junio el IMC de la institución es exactamente la fórmula. Todas las diferencias mayores a 0,05 están en **agosto**, donde el IMC se escribió a mano (máximo 0,46 kg/m²): son pequeñas frente al rango observado (≈13,6 a 40,3) y se conserva el valor registrado por la institución.

## 6. Verificación contra el consolidado anterior
El equipo había preparado antes `Nutricion_Consolidado_2026.xlsx` a mano. Se compara celda por celda. Las únicas diferencias admitidas son las fechas de nacimiento: las 8 escritas como `DD/MM/AAAA` (ahora en formato ISO) y una fecha mal transcrita.

In [7]:
if ANTIGUO.exists():
    ant = pd.read_excel(ANTIGUO, sheet_name="Consolidado")
    k = ["periodo", "IDs_FichaNominal"]
    m = nut.merge(ant, on=k, how="outer", suffixes=("", "_ant"), indicator=True)
    print("Cruce:", m["_merge"].value_counts().to_dict())
    assert (m["_merge"] == "both").all()
    for c in ["Peso_kg", "Talla_cm"]:
        assert ((m[c] - m[c+"_ant"]).abs() < 1e-9).all(), c
    for c in ["P/E_OMS","T/E_OMS","P/T_OMS","IMC_OMS","P/E_INS","T/E_INS","P/T_INS","IMC_INS"]:
        a, b = m[c].map(lambda v: "" if v is None or v != v else norm(v)), m[c+"_ant"].map(lambda v: "" if v != v or norm(v) == "*" else norm(v))
        assert (a == b).all(), (c, m.loc[a != b, k].values[:5])
    nac_ant = m["Fecha de nacimiento"].astype(str)
    dif_nac = m[m["nac"] != nac_ant][k + ["hoja", "nac", "Fecha de nacimiento"]]
    print("Peso, talla y los 8 diagnósticos: idénticos en las", len(m), "filas.")
    print("Fechas de nacimiento distintas:", len(dif_nac)); print(dif_nac.to_string(index=False))
else:
    print("No se encontró el consolidado anterior; se omite la comparación."); dif_nac = pd.DataFrame()

Cruce: {'both': 379, 'left_only': 0, 'right_only': 0}
Peso, talla y los 8 diagnósticos: idénticos en las 379 filas.
Fechas de nacimiento distintas: 9
periodo IDs_FichaNominal             hoja        nac Fecha de nacimiento
2026_01      PPA_FN_0030 MIGUEL ECHENIQUE 2012-11-21          21/11/2012
2026_01      PPA_FN_0033 MIGUEL ECHENIQUE 2014-04-12          12/04/2014
2026_01      PPA_FN_0034 MIGUEL ECHENIQUE 2014-04-12          12/04/2014
2026_01      PPA_FN_0090      TOMAS VALLE 2015-08-19          19/08/2015
2026_04      PPA_FN_0030 MIGUEL ECHENIQUE 2012-11-21          21/11/2012
2026_04      PPA_FN_0033 MIGUEL ECHENIQUE 2014-04-12          12/04/2014
2026_04      PPA_FN_0034 MIGUEL ECHENIQUE 2014-04-12          12/04/2014
2026_06      PPA_FN_0011         INFANTES 2020-03-04          2020-03-40
2026_06      PPA_FN_0030 MIGUEL ECHENIQUE 2012-11-21          21/11/2012


**Interpretación.** El consolidado nuevo reproduce exactamente las 379 filas, los pesos, tallas y los 8 diagnósticos del anterior. Las 9 diferencias son todas de fecha de nacimiento: 8 son las fechas `DD/MM/AAAA` (mismo día, ahora en formato ISO) y la novena es un **error de transcripción del consolidado anterior**: `PPA_FN_0011` en junio figuraba como `2020-03-40`, mientras que el Excel original dice `2020-03-04`. Con el consolidado anterior, esas 9 filas se perdían en el cruce con la ficha.

## 7. Guardado y reporte

In [8]:
cols = ["periodo","archivo","hoja","casa","IDs_FichaNominal","nac","Peso_kg","Talla_cm","IMC_valor","imc_recalculado",
        "Perimetro_cefalico","Presion_arterial","P/E_OMS","T/E_OMS","P/T_OMS","IMC_OMS","P/E_INS","T/E_INS","P/T_INS","IMC_INS"]
nut[cols].to_csv(OUT/"step2a_nutricion_2026.csv", sep="|", index=False, encoding="utf-8")
ctrl.to_csv(OUT/"step2a_control_origen.csv", sep="|", index=False, encoding="utf-8")
print("Guardado:", nut[cols].shape, "| control:", ctrl.shape)
nut.groupby(["periodo","casa"]).size().unstack(fill_value=0)

Guardado: (379, 20) | control: (19, 7)


casa,INFANTES,MATOGROSO HOMBRES,MATOGROSO MUJERES,MIGUEL ECHENIQUE,TOMAS VALLE
periodo,,,,,
2026_01,33,32,28,17,22
2026_02,7,0,0,0,0
2026_03,6,0,0,0,0
2026_04,30,20,15,16,20
2026_06,30,18,15,17,21
2026_08,0,17,15,0,0


| Paso | Resultado |
|---|---|
| Hojas leídas | 19 (6 archivos), mapeo explícito verificado contra cabeceras |
| Filas con ID | 402 |
| Excluidas (PROVINCIA, sin medidas) | 23 |
| Evaluaciones consolidadas | 379 |
| Fechas `DD/MM/AAAA` convertidas | 8 (4 NNA) |
| IMC recalculado | 6 (marzo) |
| Error corregido respecto al consolidado anterior | `PPA_FN_0011`, junio: `2020-03-40` → `2020-03-04` |
| Columna `Edad` | no se consolida (fórmula `HOY()` en junio y agosto) |

## Verificación y cifras de este notebook

In [9]:
assert len(ctrl) == 19 and int(ctrl["filas_id"].sum()) == 402 and int(ctrl["filas_provincia"].sum()) == 23 and len(nut) == 379
assert len(ddmm) == 8 and int(nut["imc_recalculado"].sum()) == 6
assert nut.duplicated(["periodo", "IDs_FichaNominal"]).sum() == 0
assert not ANTIGUO.exists() or len(dif_nac) == 9
assert nut.loc[(nut["IDs_FichaNominal"] == "PPA_FN_0011"), "nac"].eq("2020-03-04").all()
cifras = {"filas_id": 402, "filas_provincia": 23, "evaluaciones": len(nut), "ids": int(nut["IDs_FichaNominal"].nunique()),
          "fechas_ddmm": len(ddmm), "imc_recalculado": 6}
json.dump(cifras, open(RES/"cifras_step2a.json", "w"), indent=1); cifras

{'filas_id': 402,
 'filas_provincia': 23,
 'evaluaciones': 379,
 'ids': 140,
 'fechas_ddmm': 8,
 'imc_recalculado': 6}

## Declaración de uso de IA
Se usó **Claude Code (Anthropic)** para auditar el consolidado anterior contra los Excel originales, detectar las diferencias de columnas entre hojas y redactar el código de consolidación. El equipo validó el resultado con las verificaciones de este notebook: cabeceras de cada columna mapeada, conteos por hoja y comparación celda por celda con el consolidado anterior.

Siguiente: `Step2b_Seleccion_y_Entendimiento.ipynb`.